# Titanic exp004 - Add HasCabin

この notebook は `exp001` の真の baseline に `HasCabin` だけを追加して比較するためのものです。

仮説:
- `Cabin` 情報の有無は生存率と関係があり、`HasCabin` を足すと baseline より精度が上がる


In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    COMP_DIR = cwd.parent
elif cwd.name == "titanic":
    COMP_DIR = cwd
else:
    COMP_DIR = Path("/home/sora/dev/kaggle/competitions/titanic")

DATA_DIR = COMP_DIR / "data"
SUBMISSION_DIR = COMP_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = SUBMISSION_DIR / "exp004_hascabin_submission.csv"

FEATURE_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "HasCabin"]
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked", "HasCabin"]


In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df["HasCabin"] = train_df["Cabin"].notna().astype(int)
test_df["HasCabin"] = test_df["Cabin"].notna().astype(int)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
display(train_df.head())
display(train_df.groupby("HasCabin")["Survived"].agg(["mean", "count"]))


train shape: (891, 13)
test shape: (418, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,HasCabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0


,mean,count
HasCabin,,
0,0.299854,687
1,0.666667,204


In [3]:
summary_df = pd.DataFrame(
    {
        "dtype": train_df[FEATURE_COLUMNS + ["Survived"]].dtypes.astype(str),
        "missing": train_df[FEATURE_COLUMNS + ["Survived"]].isna().sum(),
        "missing_rate": (train_df[FEATURE_COLUMNS + ["Survived"]].isna().mean() * 100).round(2),
        "n_unique": train_df[FEATURE_COLUMNS + ["Survived"]].nunique(),
    }
).sort_values("missing", ascending=False)

display(summary_df)

,dtype,missing,missing_rate,n_unique
Age,float64,177,19.87,88
Embarked,str,2,0.22,3
Pclass,int64,0,0.00,3
SibSp,int64,0,0.00,7
Sex,str,0,0.00,2
Parch,int64,0,0.00,7
Fare,float64,0,0.00,248
HasCabin,int64,0,0.00,2
Survived,int64,0,0.00,2


## HasCabin Pipeline

使う列は baseline の 7 列に `HasCabin` を加えた 8 列です。

- `Pclass`
- `Sex`
- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Embarked`
- `HasCabin`

ここでは `Title` や `FamilySize` など他の追加特徴量はまだ使いません。


In [4]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            NUMERIC_COLUMNS,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", make_one_hot_encoder()),
                ]
            ),
            CATEGORICAL_COLUMNS,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                C=1.0,
                solver="lbfgs",
                random_state=42,
            ),
        ),
    ]
)


In [5]:
X = train_df[FEATURE_COLUMNS].copy()
y = train_df["Survived"].astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print("CV scores:", [round(score, 5) for score in scores])
print("CV mean:", round(scores.mean(), 5))
print("CV std:", round(scores.std(), 5))

CV scores: [np.float64(0.80447), np.float64(0.80899), np.float64(0.76966), np.float64(0.78652), np.float64(0.8427)]
CV mean: 0.80247
CV std: 0.02448


In [6]:
pipeline.fit(X, y)
test_predictions = pipeline.predict(test_df[FEATURE_COLUMNS].copy()).astype(int)

submission_df = pd.DataFrame(
    {
        "PassengerId": test_df["PassengerId"],
        "Survived": test_predictions,
    }
)
submission_df.to_csv(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
display(submission_df.head())


saved: /home/sora/dev/kaggle/competitions/titanic/submissions/exp004_hascabin_submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## Next Step

baseline との比較メモ:

- `exp001` CV mean: `0.79687`
- `exp004` CV mean: 実行結果を記入
- 差分: 実行結果を見て記入
